In [ ]:
"""
Stage 8 Analysis: LLM Predictions of Climate Beliefs
Replicates Stata coefplot analysis in Python
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
import seaborn as sns

!pip install -q statsmodels scikit-learn

# IMPORTANT: Use matplotlib 3.9+ syntax
matplotlib.rcParams.update({})

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ==============================================================================
# 1. DATA LOADING AND PREPARATION
# ==============================================================================

# Load predictions data
predictions = pd.read_csv('predictions_all_stages_long.csv')
predictions_stage8 = predictions[predictions['stage'] == 8].copy()
predictions_stage8 = predictions_stage8.drop('stage', axis=1)

# Load country-level inputs
inputs = pd.read_csv('country_llm_prompts_outcome2_8stages.csv')

# Merge
data = inputs.merge(predictions_stage8, on='countrynew', how='inner')

# Create outcome variables
data['actual_belief'] = data['mean_other_willingness'] * 100  # Convert to 0-100 scale
data['gpt_belief'] = data['pred_gpt']
data['claude_belief'] = data['pred_claude']
data['gemma_belief'] = data['pred_gemini']
data['llama_belief'] = data['pred_llama']

# ==============================================================================
# 2. CREATE CATEGORICAL VARIABLES
# ==============================================================================

def categorize_age(age):
    if age < 25: return 1
    elif age < 30: return 2
    elif age < 35: return 3
    elif age < 40: return 4
    elif age < 45: return 5
    elif age < 50: return 6
    else: return 7

def categorize_edu(edu):
    if edu < 0.10: return 1
    elif edu < 0.20: return 2
    elif edu < 0.30: return 3
    else: return 4

def categorize_religion(rel):
    if rel < 0.25: return 1
    elif rel < 0.50: return 2
    elif rel < 0.75: return 3
    else: return 4

def categorize_hdi(hdi):
    if hdi < 0.55: return 1
    elif hdi < 0.70: return 2
    elif hdi < 0.80: return 3
    else: return 4

def categorize_gdp(gdp):
    if gdp < 10000: return 1
    elif gdp < 25000: return 2
    elif gdp < 50000: return 3
    else: return 4

def categorize_income_ineq(ineq):
    if ineq < 0.10: return 1
    elif ineq < 0.15: return 2
    elif ineq < 0.20: return 3
    else: return 4

def categorize_wealth_ineq(ineq):
    if ineq < 0.20: return 1
    elif ineq < 0.30: return 2
    elif ineq < 0.40: return 3
    else: return 4

def categorize_temp(temp):
    if temp < 10: return 1
    elif temp < 15: return 2
    elif temp < 20: return 3
    elif temp < 25: return 4
    else: return 5

def categorize_willingness(will):
    if will < 0.25: return 1
    elif will < 0.50: return 2
    elif will < 0.75: return 3
    else: return 4

# Apply categorization
data['agecat'] = data['mean_age'].apply(categorize_age)
data['educat'] = data['mean_edu'].apply(categorize_edu)
data['religcat'] = data['mean_religion'].apply(categorize_religion)
data['hdicat'] = data['hdi_2021'].apply(categorize_hdi)
data['gdp_cat'] = data['gdp_capita_2021'].apply(categorize_gdp)
data['ineq_income_cat'] = data['top1pct_income'].apply(categorize_income_ineq)
data['ineq_wealth_cat'] = data['top1pct_wealth'].apply(categorize_wealth_ineq)
data['temp_cat'] = data['temp_mean_2010_2019'].apply(categorize_temp)
data['own_will_cat'] = data['mean_own_willingness'].apply(categorize_willingness)

# ==============================================================================
# 3. REGRESSION ANALYSIS USING OLS
# ==============================================================================

from statsmodels.formula.api import ols
from statsmodels.regression.linear_model import OLSResults

# Define formula (using categorical variables)
formula = """
Q('{}') ~ C(agecat) + C(educat) + C(religcat) + C(hdicat) + C(gdp_cat) + 
           C(ineq_income_cat) + C(ineq_wealth_cat) + C(temp_cat) + C(own_will_cat)
"""

# Run regressions
results = {}
for model_name, outcome_var in [
    ('Actual', 'actual_belief'),
    ('GPT', 'gpt_belief'),
    ('CLAUDE', 'claude_belief'),
    ('GEMMA', 'gemma_belief'),
    ('LLAMA', 'llama_belief')
]:
    model = ols(formula.format(outcome_var), data=data).fit(cov_type='HC1')  # Robust SE
    results[model_name] = model
    print(f"\n{'='*80}")
    print(f"{model_name} Model Summary")
    print(f"{'='*80}")
    print(f"R-squared: {model.rsquared:.4f}")
    print(f"Adj. R-squared: {model.rsquared_adj:.4f}")
    print(f"N: {int(model.nobs)}")

# ==============================================================================
# 4. EXTRACT COEFFICIENTS FOR PLOTTING
# ==============================================================================

def extract_coefficients(results_dict, var_pattern):
    """Extract coefficients matching a pattern from all models"""
    coef_data = []
    
    for model_name, model in results_dict.items():
        params = model.params
        conf_int = model.conf_int()
        
        # Find matching coefficients
        matching_vars = [v for v in params.index if var_pattern in v]
        
        for var in matching_vars:
            # Extract category number
            try:
                cat_num = var.split('[T.')[1].split(']')[0]
            except:
                continue
                
            coef_data.append({
                'model': model_name,
                'variable': var,
                'category': int(cat_num),
                'coefficient': params[var],
                'ci_lower': conf_int.loc[var, 0],
                'ci_upper': conf_int.loc[var, 1]
            })
    
    return pd.DataFrame(coef_data)

# ==============================================================================
# 5. COEFFICIENT PLOT FUNCTION (Stata coefplot style)
# ==============================================================================

def create_coefplot(coef_df, title, labels=None, figsize=(10, 6)):
    """
    Create a coefficient plot similar to Stata's coefplot
    Shows bars for LLMs and connected points for Actual
    """
    
    # Create figure
    fig = plt.figure()
    fig.set_size_inches(figsize[0], figsize[1])
    fig.set_dpi(150)
    ax = plt.gca()
    
    # Get unique categories
    categories = sorted(coef_df['category'].unique())
    n_cats = len(categories)
    
    # Set up x positions
    x_pos = np.arange(n_cats)
    bar_width = 0.15
    
    # Define colors
    colors = {
        'GPT': '#4A90E2',
        'CLAUDE': '#E85D75',
        'GEMMA': '#50C878',
        'LLAMA': '#F4A460',
        'Actual': '#9370DB'
    }
    
    # Plot bars for LLMs
    for i, model in enumerate(['GPT', 'CLAUDE', 'GEMMA', 'LLAMA']):
        model_data = coef_df[coef_df['model'] == model].sort_values('category')
        
        offset = (i - 1.5) * bar_width
        positions = x_pos + offset
        
        # Plot bars
        ax.bar(positions, model_data['coefficient'], 
               width=bar_width, 
               label=model,
               color=colors[model],
               alpha=0.6,
               edgecolor='black',
               linewidth=0.5)
        
        # Add error bars
        yerr = np.array([
            model_data['coefficient'] - model_data['ci_lower'],
            model_data['ci_upper'] - model_data['coefficient']
        ])
        ax.errorbar(positions, model_data['coefficient'], 
                   yerr=yerr,
                   fmt='none',
                   ecolor='black',
                   capsize=3,
                   capthick=1,
                   alpha=0.5)
    
    # Plot connected points for Actual
    actual_data = coef_df[coef_df['model'] == 'Actual'].sort_values('category')
    ax.plot(x_pos, actual_data['coefficient'], 
            marker='o', 
            linestyle='--',
            linewidth=2,
            markersize=8,
            color=colors['Actual'],
            label='Actual',
            zorder=10)
    
    # Add error bars for Actual
    yerr_actual = np.array([
        actual_data['coefficient'] - actual_data['ci_lower'],
        actual_data['ci_upper'] - actual_data['coefficient']
    ])
    ax.errorbar(x_pos, actual_data['coefficient'],
               yerr=yerr_actual,
               fmt='none',
               ecolor=colors['Actual'],
               capsize=4,
               capthick=1.5,
               alpha=0.7,
               zorder=10)
    
    # Formatting
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
    ax.set_xlabel('Category', fontsize=11, fontweight='bold')
    ax.set_ylabel('Coefficient', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    
    # Set x-axis labels
    if labels:
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, fontsize=9)
    else:
        ax.set_xticks(x_pos)
        ax.set_xticklabels(categories, fontsize=9)
    
    ax.legend(loc='best', frameon=True, fontsize=9, ncol=5)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

# ==============================================================================
# 6. CREATE ALL COEFFICIENT PLOTS
# ==============================================================================

plots = {}

# Age categories
age_coef = extract_coefficients(results, 'C(agecat)')
age_labels = ['<25', '25-30', '30-35', '35-40', '40-45', '45-50', '50+']
age_labels_used = [age_labels[i-1] for i in sorted(age_coef['category'].unique())]
plots['age'] = create_coefplot(age_coef, 'Age Categories', labels=age_labels_used)
plots['age'].savefig('coefplot_age_stage8.png', dpi=300, bbox_inches='tight')

# Education categories
edu_coef = extract_coefficients(results, 'C(educat)')
edu_labels = ['<10%', '10-20%', '20-30%', '30%+']
edu_labels_used = [edu_labels[i-1] for i in sorted(edu_coef['category'].unique())]
plots['edu'] = create_coefplot(edu_coef, 'Education Level (% Tertiary)', labels=edu_labels_used)
plots['edu'].savefig('coefplot_edu_stage8.png', dpi=300, bbox_inches='tight')

# Religion categories
relig_coef = extract_coefficients(results, 'C(religcat)')
relig_labels = ['<25%', '25-50%', '50-75%', '75%+']
relig_labels_used = [relig_labels[i-1] for i in sorted(relig_coef['category'].unique())]
plots['relig'] = create_coefplot(relig_coef, 'Religious Importance', labels=relig_labels_used)
plots['relig'].savefig('coefplot_religion_stage8.png', dpi=300, bbox_inches='tight')

# HDI categories
hdi_coef = extract_coefficients(results, 'C(hdicat)')
hdi_labels = ['Low\n(<0.55)', 'Medium\n(0.55-0.70)', 'High\n(0.70-0.80)', 'Very High\n(0.80+)']
hdi_labels_used = [hdi_labels[i-1] for i in sorted(hdi_coef['category'].unique())]
plots['hdi'] = create_coefplot(hdi_coef, 'Human Development Index', labels=hdi_labels_used)
plots['hdi'].savefig('coefplot_hdi_stage8.png', dpi=300, bbox_inches='tight')

# GDP categories
gdp_coef = extract_coefficients(results, 'C(gdp_cat)')
gdp_labels = ['<$10k', '$10-25k', '$25-50k', '$50k+']
gdp_labels_used = [gdp_labels[i-1] for i in sorted(gdp_coef['category'].unique())]
plots['gdp'] = create_coefplot(gdp_coef, 'GDP per Capita', labels=gdp_labels_used)
plots['gdp'].savefig('coefplot_gdp_stage8.png', dpi=300, bbox_inches='tight')

# Income inequality categories
ineq_income_coef = extract_coefficients(results, 'C(ineq_income_cat)')
ineq_income_labels = ['<10%', '10-15%', '15-20%', '20%+']
ineq_income_labels_used = [ineq_income_labels[i-1] for i in sorted(ineq_income_coef['category'].unique())]
plots['ineq_income'] = create_coefplot(ineq_income_coef, 'Income Inequality (Top 1% Share)', labels=ineq_income_labels_used)
plots['ineq_income'].savefig('coefplot_income_ineq_stage8.png', dpi=300, bbox_inches='tight')

# Wealth inequality categories
ineq_wealth_coef = extract_coefficients(results, 'C(ineq_wealth_cat)')
ineq_wealth_labels = ['<20%', '20-30%', '30-40%', '40%+']
ineq_wealth_labels_used = [ineq_wealth_labels[i-1] for i in sorted(ineq_wealth_coef['category'].unique())]
plots['ineq_wealth'] = create_coefplot(ineq_wealth_coef, 'Wealth Inequality (Top 1% Share)', labels=ineq_wealth_labels_used)
plots['ineq_wealth'].savefig('coefplot_wealth_ineq_stage8.png', dpi=300, bbox_inches='tight')

# Temperature categories
temp_coef = extract_coefficients(results, 'C(temp_cat)')
temp_labels = ['<10°C', '10-15°C', '15-20°C', '20-25°C', '25°C+']
temp_labels_used = [temp_labels[i-1] for i in sorted(temp_coef['category'].unique())]
plots['temp'] = create_coefplot(temp_coef, 'Average Temperature 2010-2019', labels=temp_labels_used)
plots['temp'].savefig('coefplot_temp_stage8.png', dpi=300, bbox_inches='tight')

# Own willingness categories
own_will_coef = extract_coefficients(results, 'C(own_will_cat)')
own_will_labels = ['<25%', '25-50%', '50-75%', '75%+']
own_will_labels_used = [own_will_labels[i-1] for i in sorted(own_will_coef['category'].unique())]
plots['own_will'] = create_coefplot(own_will_coef, 'Own Willingness to Contribute', labels=own_will_labels_used)
plots['own_will'].savefig('coefplot_own_willingness_stage8.png', dpi=300, bbox_inches='tight')

print("\n✓ All coefficient plots created successfully!")

# ==============================================================================
# 7. CREATE COMBINED COEFFICIENT PLOT (3x3 grid)
# ==============================================================================

fig = plt.figure()
fig.set_size_inches(18, 16)
fig.set_dpi(150)

plot_specs = [
    (age_coef, 'Age Categories', age_labels_used),
    (edu_coef, 'Education Level', edu_labels_used),
    (relig_coef, 'Religious Importance', relig_labels_used),
    (hdi_coef, 'Human Development Index', hdi_labels_used),
    (gdp_coef, 'GDP per Capita', gdp_labels_used),
    (ineq_income_coef, 'Income Inequality', ineq_income_labels_used),
    (ineq_wealth_coef, 'Wealth Inequality', ineq_wealth_labels_used),
    (temp_coef, 'Temperature', temp_labels_used),
    (own_will_coef, 'Own Willingness', own_will_labels_used)
]

colors = {
    'GPT': '#4A90E2',
    'CLAUDE': '#E85D75',
    'GEMMA': '#50C878',
    'LLAMA': '#F4A460',
    'Actual': '#9370DB'
}

for idx, (coef_df, title, labels) in enumerate(plot_specs, 1):
    ax = plt.subplot(3, 3, idx)
    
    categories = sorted(coef_df['category'].unique())
    n_cats = len(categories)
    x_pos = np.arange(n_cats)
    bar_width = 0.15
    
    # Plot bars for LLMs
    for i, model in enumerate(['GPT', 'CLAUDE', 'GEMMA', 'LLAMA']):
        model_data = coef_df[coef_df['model'] == model].sort_values('category')
        offset = (i - 1.5) * bar_width
        positions = x_pos + offset
        
        ax.bar(positions, model_data['coefficient'], 
               width=bar_width, 
               label=model if idx == 1 else "",
               color=colors[model],
               alpha=0.6,
               edgecolor='black',
               linewidth=0.5)
    
    # Plot connected points for Actual
    actual_data = coef_df[coef_df['model'] == 'Actual'].sort_values('category')
    ax.plot(x_pos, actual_data['coefficient'], 
            marker='o', 
            linestyle='--',
            linewidth=1.5,
            markersize=6,
            color=colors['Actual'],
            label='Actual' if idx == 1 else "",
            zorder=10)
    
    # Formatting
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=8, rotation=0)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# Add legend
handles, labels_leg = plt.subplot(3, 3, 1).get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='upper center', bbox_to_anchor=(0.5, 0.98), 
          ncol=5, fontsize=11, frameon=True)

plt.suptitle('Stage 8 Coefficient Comparison: LLM Predictions vs Actual Beliefs',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('coefplot_combined_stage8.png', dpi=300, bbox_inches='tight')

print("✓ Combined coefficient plot created!")

# ==============================================================================
# 8. PREDICTION PERFORMANCE METRICS
# ==============================================================================

print("\n" + "="*80)
print("PREDICTION PERFORMANCE METRICS")
print("="*80)

# Correlations
print("\nCorrelations with Actual:")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    corr = data['actual_belief'].corr(data[model])
    print(f"{model.replace('_belief', '').upper():8s}: {corr:.4f}")

# Mean Absolute Error
print("\nMean Absolute Error (MAE):")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    mae = mean_absolute_error(data['actual_belief'], data[model])
    print(f"{model.replace('_belief', '').upper():8s}: {mae:.4f}")

# Root Mean Squared Error
print("\nRoot Mean Squared Error (RMSE):")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    rmse = np.sqrt(mean_squared_error(data['actual_belief'], data[model]))
    print(f"{model.replace('_belief', '').upper():8s}: {rmse:.4f}")

# ==============================================================================
# 9. SCATTER PLOTS: Predicted vs Actual
# ==============================================================================

# Combined scatter plot
fig = plt.figure()
fig.set_size_inches(12, 10)
fig.set_dpi(150)
ax = plt.gca()

scatter_colors = {
    'GPT': '#4A90E2',
    'CLAUDE': '#E85D75',
    'GEMMA': '#50C878',
    'LLAMA': '#F4A460'
}

markers = {
    'GPT': 'o',
    'CLAUDE': 's',
    'GEMMA': '^',
    'LLAMA': 'D'
}

for model_key, model_full in [('gpt', 'GPT'), ('claude', 'CLAUDE'), 
                                ('gemma', 'GEMMA'), ('llama', 'LLAMA')]:
    ax.scatter(data['actual_belief'], data[f'{model_key}_belief'],
              alpha=0.5, s=80, 
              color=scatter_colors[model_full],
              marker=markers[model_full],
              label=model_full,
              edgecolors='black',
              linewidth=0.5)

# Add 45-degree line
ax.plot([0, 100], [0, 100], 'k--', linewidth=2, alpha=0.5, label='45° line')

ax.set_xlabel('Actual Belief about Others (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted Belief about Others (%)', fontsize=13, fontweight='bold')
ax.set_title('LLM Predictions vs Actual Beliefs - Stage 8\n(Demographics + Economics + Temperature + Actual Willingness)',
            fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=11, frameon=True)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('scatter_all_models_stage8.png', dpi=300, bbox_inches='tight')

print("✓ Scatter plot created!")

# Individual scatter plots
for model_key, model_full in [('gpt', 'GPT-4'), ('claude', 'Claude'), 
                                ('gemma', 'Gemma'), ('llama', 'Llama')]:
    fig = plt.figure()
    fig.set_size_inches(8, 8)
    fig.set_dpi(150)
    ax = plt.gca()
    
    color = scatter_colors[model_full.upper().replace('-4', '')]
    
    ax.scatter(data['actual_belief'], data[f'{model_key}_belief'],
              alpha=0.6, s=100, color=color, edgecolors='black', linewidth=0.5)
    
    # Add regression line
    z = np.polyfit(data['actual_belief'], data[f'{model_key}_belief'], 1)
    p = np.poly1d(z)
    ax.plot([0, 100], p([0, 100]), color=color, linewidth=2.5, alpha=0.8, label='Fitted line')
    
    # Add 45-degree line
    ax.plot([0, 100], [0, 100], 'k--', linewidth=2, alpha=0.5, label='45° line')
    
    # Calculate R²
    corr = data['actual_belief'].corr(data[f'{model_key}_belief'])
    r_squared = corr ** 2
    
    ax.text(0.05, 0.95, f'R² = {r_squared:.4f}', 
           transform=ax.transAxes, fontsize=12, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Actual Belief about Others (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Predicted Belief about Others (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_full} Predictions - Stage 8', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig(f'scatter_{model_key}_stage8.png', dpi=300, bbox_inches='tight')

print("✓ Individual scatter plots created!")

# ==============================================================================
# 10. SUMMARY STATISTICS TABLE
# ==============================================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

summary_cols = ['actual_belief', 'gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']
summary_stats = data[summary_cols].describe()
print("\n", summary_stats.round(2))

# Prediction errors
for model_key in ['gpt', 'claude', 'gemma', 'llama']:
    data[f'{model_key}_error'] = data[f'{model_key}_belief'] - data['actual_belief']

error_cols = ['gpt_error', 'claude_error', 'gemma_error', 'llama_error']
print("\n" + "="*80)
print("PREDICTION ERRORS (Predicted - Actual)")
print("="*80)
print("\n", data[error_cols].describe().round(2))

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  - coefplot_*.png (individual coefficient plots)")
print("  - coefplot_combined_stage8.png (3x3 grid)")
print("  - scatter_*.png (scatter plots)")

In [13]:
"""
Stage 8 Analysis: LLM Predictions of Climate Beliefs
Simplified version using sklearn (no statsmodels required)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelBinarizer
import seaborn as sns

# IMPORTANT: Use matplotlib 3.9+ syntax
matplotlib.rcParams.update({})

!pip install -q statsmodels scikit-learn

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("STAGE 8 ANALYSIS - SIMPLIFIED VERSION")
print("="*80)

# ==============================================================================
# 1. DATA LOADING AND PREPARATION
# ==============================================================================

print("\n1. Loading data...")

# Load predictions data
predictions = pd.read_csv('predictions_all_stages_long.csv')
predictions_stage8 = predictions[predictions['stage'] == 8].copy()
predictions_stage8 = predictions_stage8.drop('stage', axis=1)

# Load country-level inputs
inputs = pd.read_csv('country_llm_prompts_outcome2_8stages.csv')

# Merge
data = inputs.merge(predictions_stage8, on='countrynew', how='inner')

# Create outcome variables
data['actual_belief'] = data['mean_other_willingness'] * 100  # Convert to 0-100 scale
data['gpt_belief'] = data['pred_gpt']
data['claude_belief'] = data['pred_claude']
data['gemma_belief'] = data['pred_gemini']
data['llama_belief'] = data['pred_llama']

print(f"Loaded data for {len(data)} countries")

# ==============================================================================
# 2. CREATE CATEGORICAL VARIABLES
# ==============================================================================

print("\n2. Creating categorical variables...")

def categorize_age(age):
    if age < 30: return '< 30'
    elif age < 40: return '30-40'
    elif age < 50: return '40-50'
    else: return '50+'

def categorize_edu(edu):
    if edu < 0.10: return '< 10%'
    elif edu < 0.20: return '10-20%'
    elif edu < 0.30: return '20-30%'
    else: return '30%+'

def categorize_religion(rel):
    if rel < 0.50: return '< 50%'
    elif rel < 0.75: return '50-75%'
    else: return '75%+'

def categorize_hdi(hdi):
    if hdi < 0.70: return 'Low-Medium'
    elif hdi < 0.80: return 'High'
    else: return 'Very High'

def categorize_gdp(gdp):
    if gdp < 10000: return '< $10k'
    elif gdp < 25000: return '$10-25k'
    elif gdp < 50000: return '$25-50k'
    else: return '$50k+'

def categorize_temp(temp):
    if temp < 15: return '< 15°C'
    elif temp < 20: return '15-20°C'
    elif temp < 25: return '20-25°C'
    else: return '25°C+'

def categorize_willingness(will):
    if will < 0.50: return '< 50%'
    elif will < 0.75: return '50-75%'
    else: return '75%+'

def categorize_income_ineq(ineq):
    if ineq < 0.10: return '< 10%'
    elif ineq < 0.15: return '10-15%'
    elif ineq < 0.20: return '15-20%'
    else: return '20%+'

def categorize_wealth_ineq(ineq):
    if ineq < 0.20: return '< 20%'
    elif ineq < 0.30: return '20-30%'
    elif ineq < 0.40: return '30-40%'
    else: return '40%+'

# Apply categorization
data['age_cat_label'] = data['mean_age'].apply(categorize_age)
data['edu_cat_label'] = data['mean_edu'].apply(categorize_edu)
data['relig_cat_label'] = data['mean_religion'].apply(categorize_religion)
data['hdi_cat_label'] = data['hdi_2021'].apply(categorize_hdi)
data['gdp_cat_label'] = data['gdp_capita_2021'].apply(categorize_gdp)
data['temp_cat_label'] = data['temp_mean_2010_2019'].apply(categorize_temp)
data['will_cat_label'] = data['mean_own_willingness'].apply(categorize_willingness)
data['ineq_income_cat_label'] = data['top1pct_income'].apply(categorize_income_ineq)
data['ineq_wealth_cat_label'] = data['top1pct_wealth'].apply(categorize_wealth_ineq)

print("Categorical variables created!")

# ==============================================================================
# 3. CALCULATE MEAN PREDICTIONS BY CATEGORY
# ==============================================================================

print("\n3. Calculating mean predictions by category...")

def get_category_means(data, cat_column, outcome_cols):
    """Calculate mean outcomes for each category"""
    means = data.groupby(cat_column)[outcome_cols].mean()
    counts = data.groupby(cat_column).size()
    
    # Calculate standard errors
    sems = data.groupby(cat_column)[outcome_cols].sem()
    
    return means, sems, counts

# Outcome columns
outcome_cols = ['actual_belief', 'gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']

# Calculate means for each categorical variable
categories = {
    'Age': ('age_cat_label', ['< 30', '30-40', '40-50', '50+']),
    'Education': ('edu_cat_label', ['< 10%', '10-20%', '20-30%', '30%+']),
    'Religion': ('relig_cat_label', ['< 50%', '50-75%', '75%+']),
    'HDI': ('hdi_cat_label', ['Low-Medium', 'High', 'Very High']),
    'GDP': ('gdp_cat_label', ['< $10k', '$10-25k', '$25-50k', '$50k+']),
    'Income Inequality': ('ineq_income_cat_label', ['< 10%', '10-15%', '15-20%', '20%+']),
    'Wealth Inequality': ('ineq_wealth_cat_label', ['< 20%', '20-30%', '30-40%', '40%+']),
    'Temperature': ('temp_cat_label', ['< 15°C', '15-20°C', '20-25°C', '25°C+']),
    'Own Willingness': ('will_cat_label', ['< 50%', '50-75%', '75%+'])
}

category_stats = {}
for name, (col, order) in categories.items():
    means, sems, counts = get_category_means(data, col, outcome_cols)
    # Reorder according to specified order
    means = means.reindex([o for o in order if o in means.index])
    sems = sems.reindex([o for o in order if o in sems.index])
    category_stats[name] = (means, sems, counts)

print("Category means calculated!")

# ==============================================================================
# 4. CREATE COEFFICIENT-STYLE PLOTS
# ==============================================================================

print("\n4. Creating coefficient plots...")

def create_category_plot(means_df, sems_df, title, figsize=(10, 6)):
    """Create a plot showing means by category (similar to coefficient plot)"""
    
    fig = plt.figure()
    fig.set_size_inches(figsize[0], figsize[1])
    fig.set_dpi(150)
    ax = plt.gca()
    
    categories = means_df.index.tolist()
    n_cats = len(categories)
    x_pos = np.arange(n_cats)
    bar_width = 0.15
    
    # Define colors
    colors = {
        'gpt_belief': '#4A90E2',
        'claude_belief': '#E85D75',
        'gemma_belief': '#50C878',
        'llama_belief': '#F4A460',
        'actual_belief': '#9370DB'
    }
    
    model_labels = {
        'gpt_belief': 'GPT',
        'claude_belief': 'CLAUDE',
        'gemma_belief': 'GEMMA',
        'llama_belief': 'LLAMA',
        'actual_belief': 'Actual'
    }
    
    # Subtract baseline (first category) to show relative effects
    baseline = means_df.iloc[0]
    relative_means = means_df - baseline
    
    # Plot bars for LLMs
    llm_cols = ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']
    for i, col in enumerate(llm_cols):
        offset = (i - 1.5) * bar_width
        positions = x_pos + offset
        
        ax.bar(positions, relative_means[col], 
               width=bar_width, 
               label=model_labels[col],
               color=colors[col],
               alpha=0.6,
               edgecolor='black',
               linewidth=0.5)
        
        # Add error bars (95% CI ≈ 1.96 * SEM)
        yerr = 1.96 * sems_df[col]
        ax.errorbar(positions, relative_means[col], 
                   yerr=yerr,
                   fmt='none',
                   ecolor='black',
                   capsize=3,
                   capthick=1,
                   alpha=0.5)
    
    # Plot connected points for Actual
    ax.plot(x_pos, relative_means['actual_belief'], 
            marker='o', 
            linestyle='--',
            linewidth=2,
            markersize=8,
            color=colors['actual_belief'],
            label=model_labels['actual_belief'],
            zorder=10)
    
    # Add error bars for Actual
    yerr_actual = 1.96 * sems_df['actual_belief']
    ax.errorbar(x_pos, relative_means['actual_belief'],
               yerr=yerr_actual,
               fmt='none',
               ecolor=colors['actual_belief'],
               capsize=4,
               capthick=1.5,
               alpha=0.7,
               zorder=10)
    
    # Formatting
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
    ax.set_xlabel('Category', fontsize=11, fontweight='bold')
    ax.set_ylabel('Relative Effect (vs. baseline)', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(categories, fontsize=9, rotation=15, ha='right')
    ax.legend(loc='best', frameon=True, fontsize=9, ncol=5)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

# Create individual plots
for name, (means, sems, counts) in category_stats.items():
    fig = create_category_plot(means, sems, f'{name} - Stage 8')
    filename_base = f"coefplot_{name.lower().replace(' ', '_')}_stage8"
    fig.savefig(f"{filename_base}.png", dpi=300, bbox_inches='tight')
    fig.savefig(f"{filename_base}.pdf", bbox_inches='tight')
    print(f"  ✓ Created {filename_base}.png and .pdf")
    plt.close()

print("All coefficient plots created!")

# ==============================================================================
# 5. CREATE COMBINED PLOT
# ==============================================================================

print("\n5. Creating combined coefficient plot...")

fig = plt.figure()
fig.set_size_inches(20, 16)
fig.set_dpi(150)

colors = {
    'gpt_belief': '#4A90E2',
    'claude_belief': '#E85D75',
    'gemma_belief': '#50C878',
    'llama_belief': '#F4A460',
    'actual_belief': '#9370DB'
}

for idx, (name, (means, sems, counts)) in enumerate(category_stats.items(), 1):
    ax = plt.subplot(3, 3, idx)
    
    categories_list = means.index.tolist()
    n_cats = len(categories_list)
    x_pos = np.arange(n_cats)
    bar_width = 0.15
    
    # Subtract baseline
    baseline = means.iloc[0]
    relative_means = means - baseline
    
    # Plot bars for LLMs with error bars
    llm_cols = ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']
    for i, col in enumerate(llm_cols):
        offset = (i - 1.5) * bar_width
        positions = x_pos + offset
        
        ax.bar(positions, relative_means[col], 
               width=bar_width, 
               label=col.replace('_belief', '').upper() if idx == 1 else "",
               color=colors[col],
               alpha=0.6,
               edgecolor='black',
               linewidth=0.5)
        
        # Add 95% CI error bars
        yerr = 1.96 * sems[col]
        ax.errorbar(positions, relative_means[col], 
                   yerr=yerr,
                   fmt='none',
                   ecolor='black',
                   capsize=2,
                   capthick=0.8,
                   alpha=0.4)
    
    # Plot connected points for Actual with error bars
    ax.plot(x_pos, relative_means['actual_belief'], 
            marker='o', 
            linestyle='--',
            linewidth=1.5,
            markersize=6,
            color=colors['actual_belief'],
            label='Actual' if idx == 1 else "",
            zorder=10)
    
    # Add 95% CI error bars for Actual
    yerr_actual = 1.96 * sems['actual_belief']
    ax.errorbar(x_pos, relative_means['actual_belief'],
               yerr=yerr_actual,
               fmt='none',
               ecolor=colors['actual_belief'],
               capsize=3,
               capthick=1,
               alpha=0.6,
               zorder=10)
    
    # Formatting
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(categories_list, fontsize=7, rotation=25, ha='right')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# Add legend
handles, labels_leg = plt.subplot(3, 3, 1).get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='upper center', bbox_to_anchor=(0.5, 0.98), 
          ncol=5, fontsize=11, frameon=True)

plt.suptitle('Stage 8: LLM Predictions vs Actual Beliefs - All 9 Predictors (with 95% CI)',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('coefplot_combined_stage8.png', dpi=300, bbox_inches='tight')
plt.savefig('coefplot_combined_stage8.pdf', bbox_inches='tight')
plt.close()

print("  ✓ Combined plot created (PNG and PDF)!")

# ==============================================================================
# 6. PREDICTION PERFORMANCE METRICS
# ==============================================================================

print("\n" + "="*80)
print("PREDICTION PERFORMANCE METRICS")
print("="*80)

# Correlations
print("\nCorrelations with Actual:")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    corr = data['actual_belief'].corr(data[model])
    print(f"{model.replace('_belief', '').upper():8s}: {corr:.4f}")

# Mean Absolute Error
print("\nMean Absolute Error (MAE):")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    mae = mean_absolute_error(data['actual_belief'], data[model])
    print(f"{model.replace('_belief', '').upper():8s}: {mae:.4f}")

# Root Mean Squared Error
print("\nRoot Mean Squared Error (RMSE):")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    rmse = np.sqrt(mean_squared_error(data['actual_belief'], data[model]))
    print(f"{model.replace('_belief', '').upper():8s}: {rmse:.4f}")

# R-squared
print("\nR-squared:")
for model in ['gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']:
    r2 = r2_score(data['actual_belief'], data[model])
    print(f"{model.replace('_belief', '').upper():8s}: {r2:.4f}")

# ==============================================================================
# 7. SCATTER PLOTS - 2x2 GRID
# ==============================================================================

print("\n" + "="*80)
print("CREATING SCATTER PLOTS")
print("="*80)

# Create 2x2 grid with all 4 models
fig = plt.figure()
fig.set_size_inches(16, 16)
fig.set_dpi(150)

scatter_colors = {
    'gpt': '#4A90E2',
    'claude': '#E85D75',
    'gemma': '#50C878',
    'llama': '#F4A460'
}

for idx, (model_key, model_full) in enumerate([('gpt', 'GPT-4'), ('claude', 'Claude'), 
                                ('gemma', 'Gemma'), ('llama', 'Llama')], 1):
    ax = plt.subplot(2, 2, idx)
    
    color = scatter_colors[model_key]
    
    ax.scatter(data['actual_belief'], data[f'{model_key}_belief'],
              alpha=0.6, s=80, color=color, edgecolors='black', linewidth=0.5)
    
    # Add regression line
    z = np.polyfit(data['actual_belief'], data[f'{model_key}_belief'], 1)
    p = np.poly1d(z)
    ax.plot([0, 100], p([0, 100]), color=color, linewidth=2.5, alpha=0.8, label='Fitted line')
    
    # Add 45-degree line
    ax.plot([0, 100], [0, 100], 'k--', linewidth=2, alpha=0.5, label='45° line')
    
    # Calculate R²
    r2 = r2_score(data['actual_belief'], data[f'{model_key}_belief'])
    
    ax.text(0.05, 0.95, f'R² = {r2:.4f}', 
           transform=ax.transAxes, fontsize=11, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Actual Belief about Others (%)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Belief about Others (%)', fontsize=11, fontweight='bold')
    ax.set_title(f'{model_full} Predictions', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)

plt.suptitle('LLM Predictions vs Actual Beliefs - Stage 8\n(Demographics + Economics + Temperature + Actual Willingness)', 
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('scatter_all_models_stage8.png', dpi=300, bbox_inches='tight')
plt.savefig('scatter_all_models_stage8.pdf', bbox_inches='tight')
plt.close()

print("  ✓ Scatter plots created as 2x2 grid (PNG and PDF)!")

# ==============================================================================
# 8. SUMMARY STATISTICS
# ==============================================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

summary_cols = ['actual_belief', 'gpt_belief', 'claude_belief', 'gemma_belief', 'llama_belief']
summary_stats = data[summary_cols].describe()
print("\n", summary_stats.round(2))

# Prediction errors
for model_key in ['gpt', 'claude', 'gemma', 'llama']:
    data[f'{model_key}_error'] = data[f'{model_key}_belief'] - data['actual_belief']

error_cols = ['gpt_error', 'claude_error', 'gemma_error', 'llama_error']
print("\n" + "="*80)
print("PREDICTION ERRORS (Predicted - Actual)")
print("="*80)
print("\n", data[error_cols].describe().round(2))

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  - coefplot_*.png and .pdf (9 individual coefficient plots)")
print("  - coefplot_combined_stage8.png and .pdf (3x3 grid with all 9 predictors)")
print("  - scatter_all_models_stage8.png and .pdf (2x2 grid showing all 4 models)")


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
STAGE 8 ANALYSIS - SIMPLIFIED VERSION

1. Loading data...
Loaded data for 125 countries

2. Creating categorical variables...
Categorical variables created!

3. Calculating mean predictions by category...
Category means calculated!

4. Creating coefficient plots...
  ✓ Created coefplot_age_stage8.png and .pdf
  ✓ Created coefplot_education_stage8.png and .pdf
  ✓ Created coefplot_religion_stage8.png and .pdf
  ✓ Created coefplot_hdi_stage8.png and .pdf
  ✓ Created coefplot_gdp_stage8.png and .pdf
  ✓ Created coefplot_income_inequality_stage8.png and .pdf
  ✓ Created coefplot_wealth_inequality_stage8.png and .pdf
  ✓ Created coefplot_temperature_stage8.png and .pdf
  ✓ Created coefplot_own_willingness_stage8.png and .pdf
All coefficient plots created!

5. Creating combined coefficient plot...
  ✓ Combined plot created (PNG and PDF)!

PREDICTION PERFORMANCE METRICS

Correlations

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>